# 1 — EDA + Leakage-Free Feature Engineering

> **Yazılı karşılığı:** [`docs/1_eda_and_feature_engineering.docx`](../docs/1_eda_and_feature_engineering.docx) — tüm metodoloji, tasarım kararları ve gerekçeleri docx'te. Bu notebook sayıları + grafikleri canlı koşturur ve kararların *neden* alındığını veri üzerinden gösterir.

## Bu notebook ne yapar?

Ham X Education tabular verisini adım adım keşfeder; her tasarım kararını (kolon drop, feature türetme, encode stratejisi) veri üzerinde sayısal olarak doğrular; ardından bu doğrulanmış kararların production formuna refactor edildiği `src/lead_priority/features/` modüllerini çağırıp final feature matrisini üretir.

Burada yapılan variance check, missing pattern teyidi ve crosstab'lar `constants.py`'a yazılan listelerin ve `derive.py`'a yazılan fonksiyonların doğrulanmış kaynağıdır. Her code cell öncesi markdown cell ne yapıldığını ve neden yapıldığını anlatır.

## Phase 0'dan miras

0. aşamada (`docs/0_synthetic_data_and_leakage.docx`) tek-kolon AUC ile leakage taraması yapıldı. Üç kolon outcome-sonrası bilgi içerdiği için scoring modelinden çıkarılacak:

| Kolon | Tek-kolon AUC | Karar |
|---|---|---|
| `Tags` | 0.92 | drop |
| `Lead Quality` | 0.80 | drop |
| `Last Notable Activity` | 0.70 | drop |

Bu aşamada bu üç kolonu hiçbir feature türetiminde kullanmayacağız.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lead_priority.settings import REPO_ROOT

RAW_CSV = REPO_ROOT / 'data' / 'Lead Scoring.csv'
LEAKAGE_REPORT = REPO_ROOT / 'artifacts' / 'leakage_report.json'
SUMMARY_JSON = REPO_ROOT / 'artifacts' / 'feature_summary.json'

raw = pd.read_csv(RAW_CSV)
print(f'shape: {raw.shape}')
print(f'positive rate (Converted=1): {raw["Converted"].mean():.4f}')
raw.head(3)

## 1. Sınıf dağılımı

Hedef değişken `Converted` (0/1). İlk kontrol: pozitif sınıf oranı modelin **stratified split** ihtiyacı duyup duymadığını söyler. %38.5 pozitif ⇒ dengesiz değil ama eğri; Phase 2'de splitler stratified kurulacak.

In [ ]:
counts = raw['Converted'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Not Converted (0)', 'Converted (1)'], counts.values, color=['#4c72b0', '#dd8452'])
for i, v in enumerate(counts.values):
    ax.text(i, v + 80, f'{v:,} ({v/len(raw):.1%})', ha='center')
ax.set_ylabel('lead count')
ax.set_title('Class balance (Converted)')
plt.tight_layout()
plt.show()
counts

## 2. Eksik veri paterni

Her kolonda kaç satır eksik? En yüksek 15 kolonu çıkarıyoruz. Yorum çerçevesi:

- **>40% missing** — kolon birincil sinyal sayılmaz; missing-indicator flag'i değerli olabilir (MAR sinyali).
- **10–40% missing** — feature olarak tutulur, missing-indicator ile zenginleştirilir.
- **<10% missing** — basit median/Unknown ile doldurulur.

`Tags` ve `Lead Quality` bu listede üst sıralarda görünecek; ama bunlar zaten leakage gerekçesiyle drop edilecek (yukarıdaki tablo).

In [ ]:
missing_rate = raw.isna().mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(missing_rate.index[::-1], missing_rate.values[::-1] * 100, color='#c44e52')
ax.set_xlabel('missing %')
ax.set_title('Top 15 columns by missing rate')
plt.tight_layout()
plt.show()
(missing_rate * 100).round(2).to_frame('missing_pct')

## 3. 'Select' placeholder problemi

Bazı kategorik kolonlarda `'Select'` değeri var. Bu kullanıcının dropdown'dan seçim yapmadığı durumu temsil ediyor — yani **gerçek bir kategori değil, gizli bir NaN**. Saymazsak `OneHotEncoder` bu değerleri ayrı bir sınıf olarak öğrenir ve gürültü ekler.

Tasarım kararı: `Specialization`, `How did you hear about X Education`, `Lead Profile`, `City` kolonlarında `'Select'` → NaN'a çevrilecek → SimpleImputer ile `'Unknown'` bucket'ına gidecek. Bu mantık `src/lead_priority/features/derive.py` ve `transformers.SelectToNaN` içinde production kodu olarak yaşıyor.

In [ ]:
select_candidates = [
    'Specialization',
    'How did you hear about X Education',
    'Lead Profile',
    'City',
]
rows = []
for col in select_candidates:
    s = raw[col]
    rows.append({
        'column': col,
        'n_select': int((s == 'Select').sum()),
        'n_true_nan': int(s.isna().sum()),
        'pct_combined': float(((s == 'Select') | s.isna()).mean() * 100),
    })
pd.DataFrame(rows)

## 4. Asymmetrique cohort — MAR mı MNAR mı?

Dört Asymmetrique kolonunun (`Activity Score`, `Profile Score`, `Activity Index`, `Profile Index`) hepsi yaklaşık **%45.65** missing. Sıkça birlikte missing oluyorlarsa bu **MAR (Missing At Random)** sinyalidir — üçüncü taraf bir scoring servisi bazı lead'ler için skor üretmemiş, hepsi aynı anda boş kalmış. MNAR olsaydı (Missing Not At Random — değerin kendisine bağlı eksiklik), kolonlar birbirinden bağımsız missing olur, overlap düşük çıkardı.

**Karar:** 4 kolonu birlikte tutuyoruz; numerik olanlar (Score) numeric branch'ta clip+impute+scale + missing indicator, kategorik olanlar (Index 'letter rating') one-hot + Unknown + missing indicator. Indicator bayrağı LGBM için kritik (impute sonrası MAR sinyalini split olarak yakalayabilir).

In [ ]:
asym_cols = [
    'Asymmetrique Activity Score',
    'Asymmetrique Profile Score',
    'Asymmetrique Activity Index',
    'Asymmetrique Profile Index',
]
asym_nan = raw[asym_cols].isna()
all_four_nan = int(asym_nan.all(axis=1).sum())
any_nan = int(asym_nan.any(axis=1).sum())
print(f'All four Asymmetrique columns NaN together: {all_four_nan} rows ({all_four_nan/len(raw):.2%})')
print(f'At least one Asymmetrique column NaN:        {any_nan} rows ({any_nan/len(raw):.2%})')
print(f'Pairwise overlap rate (all-four-NaN / any-NaN): {all_four_nan/any_nan:.4f}')
print()
print('Per-column NaN counts:')
asym_nan.sum().to_frame('n_nan')

## 5. Boolean variance check — ölü kolonların tespiti

Veri seti onlarca Yes/No kolonu içeriyor (`Magazine`, `Newspaper`, `Search`, …). Bunların bir kısmı tüm 9240 satırda aynı değeri (`'No'`) taşıyor — yani **variance=0**, sıfır bilgi. Bunları drop ediyoruz: hem LR hem LGBM için sıfır katkı, sadece regularizasyon yüzeyi.

Diğer bir kısmı **near-zero variance** (1–14 'Yes'): bireysel olarak LR'i gürültülendirir ama toplandığında 'kaç kanaldan duydu' sinyalini taşırlar. Bunları `channel_diversity_count ∈ {0..5}` altında topluyoruz, bireyselleri drop ediyoruz.

**Karar tablosu:**

| Kolon | 'Yes' sayısı | Karar |
|---|---|---|
| Magazine, Receive More Updates…, Update me on Supply Chain Content, Get updates on DM Content, I agree to pay through cheque | 0 | drop (dead) |
| Newspaper Article (2), X Education Forums (1), Newspaper (1), Digital Advertisement (4), Search (14) | 1–14 | `channel_diversity_count` altında topla, bireyselleri drop |

In [ ]:
boolean_candidates = [
    'Magazine',
    'Receive More Updates About Our Courses',
    'Update me on Supply Chain Content',
    'Get updates on DM Content',
    'I agree to pay the amount through cheque',
    'Newspaper Article',
    'X Education Forums',
    'Newspaper',
    'Digital Advertisement',
    'Search',
]
rows = []
for col in boolean_candidates:
    counts = raw[col].value_counts(dropna=False).to_dict()
    rows.append({
        'column': col,
        'n_yes': int(counts.get('Yes', 0)),
        'n_no': int(counts.get('No', 0)),
        'decision': 'drop (dead)' if counts.get('Yes', 0) == 0 else 'merge into channel_diversity_count',
    })
pd.DataFrame(rows)

## 6. Lead Source / Origin bazında dönüşüm

Bazı kanallar dönüşüm oranı yüksek (Reference, Welingak), bazıları çok düşük. Bu kolonların one-hot encode edilmesi gerek; ama uzun kuyrukta `Lead Source` ~20 farklı değer içeriyor. Sklearn'ün `OneHotEncoder(min_frequency=20)` bayrağı kuyruğu `infrequent_sklearn` bucket'ına toplar — kolon sayısı kontrol altında, LR regularizasyon sağlıklı kalır.

In [ ]:
def conv_rate(col: str) -> pd.DataFrame:
    grp = raw.groupby(col)['Converted']
    out = pd.DataFrame({
        'n': grp.size(),
        'converted_pct': (grp.mean() * 100).round(1),
    }).sort_values('n', ascending=False)
    return out

print('Lead Source — top 10 by volume:')
print(conv_rate('Lead Source').head(10))
print()
print('Lead Origin:')
print(conv_rate('Lead Origin'))

## 7. Country dağılımı — neden binary `country_is_india`

Hindistan %70 ağırlıkta; geri kalan 37 ülke birlikte %3'ten az. Top-K one-hot (India / US / UAE / Other) yapsaydık ABD'de 70, BAE'de 53 lead var — bu kovalardaki dönüşüm oranı **örneklem gürültüsünden ayırt edilemez**. LR katsayıları varyans patlamasından nasibini alır.

**Karar:** `country_is_india: bool` türetilecek (`derive_features` içinde), orijinal `Country` kolonu drop. Tek bool flag yorumlu, overfit-güvenli.

In [ ]:
country_dist = raw['Country'].fillna('(missing)').value_counts().head(10)
country_dist.to_frame('n')

## 8. Phase 0 leakage raporu — drop edilecek kolonlar

0. aşamadan miras alınan tablo. Tek-kolon AUC > 0.7 olan üç kolon scoring modelinde **kullanılmayacak** — `derive_features` ilk işinde bunları drop eder.

In [ ]:
report = json.loads(LEAKAGE_REPORT.read_text())
pd.DataFrame(report['tabular_leakage_suspects'])

## 9. `derive_features` — türetilmiş feature'ların serving-side hali

Yukarıdaki tüm gözlemler (drop listeleri, 'Select' → NaN, channel diversity, country_is_india, high/negative intent activity, `total_time_per_visit`) `src/lead_priority/features/derive.py` içinde tek bir saf fonksiyona refactor edildi: `derive_features(raw_df) → derived_df`.

Bu fonksiyon hem training (`scripts/fit_feature_pipeline.py`) hem serving (Phase 5 FastAPI) tarafından aynı kod yoluyla çağrılır — train/serve drift'i sıfır.

Şimdi çağıralım ve çıktıyı inceleyelim:

In [ ]:
from lead_priority.features import derive_features
from lead_priority.features.constants import REQUIRED_DERIVED_COLUMNS

derived = derive_features(raw)
print(f'raw shape:     {raw.shape}')
print(f'derived shape: {derived.shape}  (37 raw cols → 25 derived cols)')
print()
print("Türetilmiş kolonlar (raw'da yoktu):")
new_cols = [c for c in derived.columns if c not in raw.columns]
for c in new_cols:
    print(f'  - {c}')
print()
print('İlk 3 satırda türetilmişler:')
derived[new_cols].head(3)

### Sanity check — `total_time_per_visit` NaN-safe mi?

`TotalVisits` 137 satırda eksik. Ratio'da payda olarak kullanılan `TotalVisits`'in NaN olduğu satırlarda Inf/NaN üretmemeli. `derive_features` payda'yı `max(1, TotalVisits)` ile clamp eder; geriye kalan NaN'lar downstream median imputer'a düşer.

In [ ]:
nan_visits = raw[raw['TotalVisits'].isna()].head(5).copy()
derived_subset = derive_features(
    pd.concat([raw[~raw['TotalVisits'].isna()].iloc[:1], nan_visits]).reset_index(drop=True)
)
ratios = derived_subset['total_time_per_visit']
print('total_time_per_visit on 1 normal + 5 NaN-TotalVisits rows:')
print(ratios.tolist())
print(f'\nInf count: {np.isinf(ratios).sum()}  |  finite ratio for NaN-TotalVisits: clamps to Total Time')

## 10. Tek ortak pipeline — `build_pipeline()`

Şimdi gerçek mimari. Phase 2'de hem Logistic Regression hem LightGBM **aynı** fitted feature matrisini kullanacak (apples-to-apples lift karşılaştırması). Pipeline `src/lead_priority/features/pipeline.py` içinde:

```
ColumnTransformer:
  ├── num_clipped : PercentileClipper(q=0.95) → SimpleImputer(median, add_indicator=True) → StandardScaler
  ├── num_ratio   : SimpleImputer(median) → StandardScaler         (zaten bounded)
  ├── cat         : SelectToNaN → SimpleImputer('Unknown') → OneHotEncoder(min_frequency=20, handle_unknown='ignore')
  ├── country     : passthrough (country_is_india)
  └── binaries    : passthrough (Do Not Email, Do Not Call, …, is_high_intent_activity, is_negative_activity)
```

Neden tek pipeline iki modeli birden tatmin eder:
- **LR**: scale edilmiş + impute edilmiş numerik + one-hot ⇒ pipeline tamamen sağlıyor.
- **LGBM**: native NaN handle edebilir ama impute ettiğimiz için bypass ediliyor. `add_indicator=True` ile fit zamanı NaN gören her numerik için missing indicator binary çıkıyor ⇒ LGBM Asymmetrique %45.65 MAR cohort'undaki sinyali split olarak yakalar.

In [ ]:
from lead_priority.features import build_pipeline

pipeline = build_pipeline()
X = pipeline.fit_transform(derived)
feature_names = list(pipeline.named_steps['features'].get_feature_names_out())
print(f'X shape: {X.shape}')
print(f'NaN count: {int(np.isnan(X).sum())}  |  Inf count: {int(np.isinf(X).sum())}')
print(f'Total output features: {len(feature_names)}')
print()
print('İlk 10 feature ismi:')
for name in feature_names[:10]:
    print(f'  {name}')
print('...')
print('Son 5 feature ismi:')
for name in feature_names[-5:]:
    print(f'  {name}')

## 11. Feature compozisyonunun özeti

`scripts/fit_feature_pipeline.py` çalıştırıldığında `artifacts/feature_summary.json` üretilir. Bu JSON her output feature'ı ham kolon kaynağı + tipiyle birlikte listeler. Phase 2 ve sonrası bu özeti referans olarak kullanır.

In [ ]:
if SUMMARY_JSON.exists():
    summary = json.loads(SUMMARY_JSON.read_text())
    from collections import Counter
    kind_counts = Counter(f['kind'] for f in summary['features'])
    print(f"schema_version: {summary['schema_version']}")
    print(f"n_features:     {summary['n_features']}")
    print(f"n_rows_fit:     {summary['n_rows_fit']}")
    print()
    print('feature kind dağılımı:')
    for kind, n in sorted(kind_counts.items(), key=lambda x: -x[1]):
        print(f'  {kind:25s} {n:>3d}')
    print()
    print('Drop edilen kolonlar:')
    for category, cols in summary['dropped_columns'].items():
        print(f'  {category}: {cols}')
else:
    print(f'{SUMMARY_JSON} yok — önce `python scripts/fit_feature_pipeline.py` çalıştırın.')

## 12. Sonuç ve sonraki adım

- Ham 9.240 × 37 tablo → türetilmiş 9.240 × 25 DataFrame → fit edilmiş 9.240 × 104 numerik feature matrisi.
- 3 leakage kolonu, 2 PII, 5 dead boolean, 5 near-zero variance pazarlama kanalı, ve `Country` ham hali kullanım dışı.
- 5 türetilmiş feature: `total_time_per_visit`, `channel_diversity_count`, `country_is_india`, `is_high_intent_activity`, `is_negative_activity`.
- 4 missing-indicator (TotalVisits, Page Views Per Visit, Asymmetrique Activity Score, Asymmetrique Profile Score). LGBM bunlarla MAR sinyalini yakalar.
- One-hot uzun kuyruğu `min_frequency=20` ile `infrequent_sklearn` bucket'ına toplandı.

**Sonraki aşama (Phase 2):** aynı fitted pipeline'a Logistic Regression baseline ve LightGBM bağlanır. İki model aynı 104-boyutlu feature uzayında karşılaştırılır (AUC, lift, calibration, PR trade-off). `artifacts/feature_pipeline.joblib` Phase 2 ve sonrası için sabit girdi.